In [11]:
##pip install nltk
##3.10.3
##!pip install bert-score
##!pip install rouge_score

In [4]:
import nltk

nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\sowraj\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\sowraj\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [12]:
from bert_score import score as bert_score
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score
import pandas as pd
# 1. LOAD DATA
df = pd.read_csv("ai_teacher_results.csv")

# 2. PREPARE REFERENCE AND GENERATED ANSWERS
references = (
    df["Teacher Response"]
    .fillna("")
    .astype(str)
    .tolist()
)
candidates = (
    df["GeneratedAnswer"]
    .fillna("")
    .astype(str)
    .tolist()
)

# 3. BERTSCORE
P, R, F1 = bert_score(
    candidates,
    references,
    lang="en",
    verbose=True
)
df["BERT"] = F1.cpu().numpy()

# 4. ROUGE
rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"],use_stemmer=True)
rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

for candidate, reference in zip(candidates, references):
    scores = rouge.score(reference, candidate)
    rouge1_scores.append(scores["rouge1"].fmeasure)
    rouge2_scores.append(scores["rouge2"].fmeasure)
    rougeL_scores.append(scores["rougeL"].fmeasure)

df["ROUGE-1"] = rouge1_scores
df["ROUGE-2"] = rouge2_scores
df["ROUGE-L"] = rougeL_scores

# 5. METEOR
meteor_scores = []

for candidate, reference in zip(candidates, references):
    candidate_tokens = candidate.split()
    reference_tokens = reference.split()
    score = meteor_score([reference_tokens], candidate_tokens)
    meteor_scores.append(score)

df["METEOR"] = meteor_scores

# 6. ROUND SCORES

similarity_columns = [
    "BERT",
    "ROUGE-1",
    "ROUGE-2",
    "ROUGE-L",
    "METEOR"
]
df[similarity_columns] = (df[similarity_columns].round(3))

# 7. QUESTION-WISE SIMILARITY DATAFRAME

similarity_result = df[
    [
        "User Query",
        "BERT",
        "ROUGE-1",
        "ROUGE-2",
        "ROUGE-L",
        "METEOR"
    ]
].copy()

# DISPLAY QUESTION-WISE DATAFRAME

display(similarity_result)

# 8. CALCULATE AVERAGES

average_bert = df["BERT"].mean()
average_rouge1 = df["ROUGE-1"].mean()
average_rouge2 = df["ROUGE-2"].mean()
average_rougeL = df["ROUGE-L"].mean()
average_meteor = df["METEOR"].mean()

# 9. CREATE AVERAGE SIMILARITY DATAFRAME

average_similarity = pd.DataFrame({
    "Metric": ["BERT", "ROUGE-1", "ROUGE-2", "ROUGE-L", "METEOR"],
    "Average Score": [average_bert, average_rouge1, average_rouge2, average_rougeL, average_meteor]
})

# ROUND AVERAGE SCORES
average_similarity["Average Score"] = (average_similarity["Average Score"].round(3))

# DISPLAY AVERAGE DATAFRAME
display(average_similarity)

C:\anaconda 3\envs\agenticai\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sowraj\.cache\huggingface\hub\models--roberta-large. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 3218.64it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-lar

calculating scores...
computing bert embedding.


100%|██████████| 1/1 [00:08<00:00,  8.44s/it]


computing greedy matching.


100%|██████████| 1/1 [00:00<00:00, 80.29it/s]


done in 8.47 seconds, 2.36 sentences/sec


,User Query,BERT,ROUGE-1,ROUGE-2,ROUGE-L,METEOR
0,What is AI?,0.885,0.411,0.113,0.247,0.312
1,How does AI differ from traditional programming?,0.889,0.417,0.229,0.222,0.484
2,What is Machine Learning?,0.897,0.462,0.289,0.359,0.474
3,What is Deep Learning?,0.891,0.421,0.216,0.316,0.355
4,"What is the difference between AI, ML, and Dee...",0.894,0.516,0.264,0.473,0.549
5,What is training data?,0.875,0.400,0.192,0.373,0.346
6,What is a Machine Learning model?,0.905,0.459,0.306,0.459,0.537
7,What is supervised learning?,0.865,0.314,0.118,0.286,0.262
8,What is unsupervised learning?,0.893,0.384,0.085,0.274,0.267
9,What is reinforcement learning?,0.896,0.328,0.062,0.209,0.268


,Metric,Average Score
0,BERT,0.889
1,ROUGE-1,0.396
2,ROUGE-2,0.168
3,ROUGE-L,0.313
4,METEOR,0.358


In [13]:
df.to_csv(
    "ai_teacher_evaluation_Similarity_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

In [14]:
average_similarity.to_csv("ai_teacher_evaluation_Similarity_Average.csv", index=False,
    encoding="utf-8-sig")